In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from limb_fitting import *

In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')

xu, yu = s['xu'], s['yu']

In [3]:
import fnmatch
from connect import bob

np.set_printoptions(suppress=True)
sftp = bob()

top_dir = '/data/solo/phi/data/fmyy/l1/'
dirs = sorted(sftp.listdir(top_dir))

output_file = 'disk_centers_cor.csv'
with open(output_file, 'w') as f:
    f.write('date, did, xd_sun, yd_sun, rd_sun, xu_sun, yu_sun, ru_sun\n')


In [4]:
for directory in dirs:
    if directory[:4] in ['2024', '2025']:
        for file in sorted(sftp.listdir(top_dir + directory)):

            if fnmatch.fnmatch(file, '*fdt-[i,a]lam*.fits.gz'):
                try:
                    remote_file = top_dir + directory + '/' + file
                    local_file = 'temp.fits.gz'
                    
                    sftp.get(remote_file, local_file)
                    
                    with fits.open(local_file) as hdul:
                        image = hdul[0].data[0]
                        header = hdul[0].header

                    nx, ny = header['NAXIS2'], header['NAXIS1']
                    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
                    xu_, yu_ = xu[x0:x0+nx, y0:y0+ny] - x0, yu[x0:x0+nx, y0:y0+ny] - y0

                    edges = find_edges(image)
                    x, y = np.where(edges)
                    x, y = filter_outliers(x, y, acc=1)
                    xd_sun, yd_sun, rd_sun = fitnp(x, y)

                    x, y = xu_[edges], yu_[edges]
                    x, y = filter_outliers(x, y, acc=1)
                    xu_sun, yu_sun, ru_sun = fitnp(x, y)

                    date = file.split('_')[3]
                    did = file.split('_')[5].split('.')[0]

                    with open(output_file, 'a') as f:
                        f.write(f'{date}, {did}, {xd_sun:.4f}, {yd_sun:.4f}, {rd_sun:.4f}, {xu_sun:.4f}, {yu_sun:.4f}, {ru_sun:.4f}\n')
                        
                except:
                    print(file)
                    continue

sftp.close()

solo_L1_phi-fdt-alam_20251203T062003_V202602210648I_0552030502.fits.gz
solo_L1_phi-fdt-alam_20251220T092009_V202602210648I_0552200504.fits.gz
solo_L1_phi-fdt-alam_20251222T064009_V202602210648I_0552220501.fits.gz
solo_L1_phi-fdt-alam_20251225T002009_V202602210649I_0552250501.fits.gz
